In [22]:
import argparse
from concurrent.futures import ThreadPoolExecutor
import csv
import os
from pathlib import Path
import time
import zipfile
import numpy as np
import pandas as pd
import requests
from scipy.optimize import minimize

_K_SMALL = np.arange(-4, 5, dtype=float)
_K_LARGE = np.arange(1, 9, dtype=float)
_SIN_LARGE = np.where((_K_LARGE.astype(int) // 2) % 2 == 0, 1.0, -1.0)


def wfpt_pdf(rt, choice, v, a, ter=0.0):
    rt = np.asarray(rt, dtype=float)
    choice = np.broadcast_to(np.asarray(choice, dtype=int), rt.shape)
    v = np.broadcast_to(np.asarray(v, dtype=float), rt.shape)
    a = np.broadcast_to(np.asarray(a, dtype=float), rt.shape)
    ter = np.broadcast_to(np.asarray(ter, dtype=float), rt.shape)

    t = rt - ter
    out = np.zeros_like(rt, dtype=float)
    valid = (t > 1e-6) & (a > 1e-4)
    if not np.any(valid):
        return out

    tv, vv, av = t[valid], v[valid], a[valid]
    sign = np.where(choice[valid] == 1, 1.0, -1.0)
    tt = tv / (av * av)

    f_star = np.empty_like(tt)
    small = tt < 0.25
    if np.any(small):
        ts = tt[small][:, None]
        w_term = 0.5 + 2.0 * _K_SMALL[None, :]
        s_sum = np.sum(w_term * np.exp(-(w_term * w_term) / (2.0 * ts)), axis=1)
        f_star[small] = s_sum / np.sqrt(2.0 * np.pi * (tt[small] ** 3))

    large = ~small
    if np.any(large):
        tl = tt[large][:, None]
        l_sum = np.sum(
            _K_LARGE[None, :] * _SIN_LARGE[None, :] * np.exp(-0.5 * (_K_LARGE[None, :] ** 2) * (np.pi ** 2) * tl),
            axis=1,
        )
        f_star[large] = np.pi * l_sum

    log_pref = sign * 0.5 * vv * av - 0.5 * (vv * vv) * tv - 2.0 * np.log(av)
    out[valid] = np.exp(np.clip(log_pref, -700.0, 50.0)) * np.maximum(f_star, 0.0)
    return out


def wfpt_nll(rt, choice, v, a, ter, rt_span=2.0, p_lapse=0.005):
    pdf = wfpt_pdf(rt, choice, v, a, ter)
    return -np.log(np.maximum((1.0 - p_lapse) * pdf + p_lapse / rt_span, 1e-300))


def osf_walk_files(url, prefix="", depth=0):
    if depth > 3:
        return []
    items = []
    while url:
        r = requests.get(url, timeout=30)
        r.raise_for_status()
        payload = r.json()
        for entry in payload.get("data", []):
            attrs = entry["attributes"]
            path = f"{prefix}/{attrs['name']}".lstrip("/")
            if attrs["kind"] == "folder":
                sub_url = entry["relationships"]["files"]["links"]["related"]["href"]
                items.extend(osf_walk_files(sub_url, path, depth + 1))
            else:
                items.append((path, entry["links"]["download"]))
        url = payload.get("links", {}).get("next")
    return items


def ensure_osf_node(node_id, data_dir):
    node_dir = Path(data_dir) / f"osf_{node_id}"
    if node_dir.exists() and any(node_dir.rglob("*.*")):
        return node_dir

    node_dir.mkdir(parents=True, exist_ok=True)
    files = osf_walk_files(f"https://api.osf.io/v2/nodes/{node_id}/files/osfstorage/")
    for rel_path, dl_url in files:
        dest = node_dir / rel_path
        dest.parent.mkdir(parents=True, exist_ok=True)
        with requests.get(dl_url, stream=True, timeout=90) as r:
            r.raise_for_status()
            with open(dest, "wb") as f:
                for chunk in r.iter_content(chunk_size=1 << 16):
                    if chunk:
                        f.write(chunk)
        if dest.suffix.lower() == ".zip":
            with zipfile.ZipFile(dest) as zf:
                zf.extractall(dest.parent)
    return node_dir


def add_conflict_trace(df, lam=0.5):
    out = df.copy()
    traces = np.zeros(len(out), dtype=float)
    for _, idx in out.groupby(["subj", "block"], sort=False).groups.items():
        cong = out.loc[idx, "I"].to_numpy(dtype=float)
        s = np.zeros_like(cong)
        for t in range(1, len(cong)):
            s[t] = lam * s[t - 1] + 0.5 * (1.0 - cong[t - 1])
        traces[idx] = s
    out["S_t"] = traces
    out["S_bar"] = out.groupby("subj")["S_t"].transform("mean")
    return out


def add_flags(df):
    out = df.copy()
    out["trial_idx"] = out.groupby("subj").cumcount()
    n_subj = out.groupby("subj")["trial_idx"].transform("count")
    out["trial_frac"] = out["trial_idx"] / np.maximum(n_subj, 1)

    grp = out.groupby(["subj", "block"], sort=False)
    out["prev_correct"] = grp["correct"].shift(1)
    if "resp" in out.columns and out["resp"].notna().any():
        prev_resp = grp["resp"].shift(1)
        valid_pair = out["resp"].notna() & prev_resp.notna()
        out["resp_alt"] = np.where(valid_pair, (out["resp"] != prev_resp).astype(float), np.nan)
    else:
        out["resp_alt"] = np.nan
    return out


def load_hedge_flanker(data_dir, lam=0.5):
    node_dir = ensure_osf_node("cwzds", data_dir)
    cells = {}
    for session_tag, pattern in (("S1", "*Flanker*S1*.csv"), ("S2", "*Flanker*S2*.csv")):
        matches = list(node_dir.rglob(pattern))
        if not matches:
            raise FileNotFoundError(f"Missing {pattern} under {node_dir}")
        raw = pd.read_csv(matches[0])
        cols = {c.lower(): c for c in raw.columns}

        subj_col = cols.get("participant", cols.get("subject", cols.get("subj")))
        blk_col = cols.get("block", None)
        cond_col = cols.get("condition", cols.get("congruency"))
        corr_col = cols.get("correct", cols.get("accuracy"))
        rt_col = cols.get("rt", cols.get("response_time"))
        resp_col = cols.get("response", cols.get("resp", None))

        cond_num = pd.to_numeric(raw[cond_col], errors="coerce")
        if cond_num.notna().mean() > 0.8:
            # Hedge numeric coding: 0 = congruent, 1 = neutral, 2 = incongruent
            cong = np.where(cond_num == 0, 1.0, np.where(cond_num == 2, -1.0, np.nan))
        else:
            c_str = raw[cond_col].astype(str).str.strip().str.lower()
            is_inc = c_str.str.contains("incong|incomp")
            is_con = c_str.str.contains("^cong|^comp") & ~is_inc
            cong = np.where(is_con, 1.0, np.where(is_inc, -1.0, np.nan))

        rt = pd.to_numeric(raw[rt_col], errors="coerce")
        if rt.median() > 10.0:
            rt = rt / 1000.0

        resp_series = raw[resp_col].where(raw[resp_col].notna(), np.nan) if resp_col else pd.Series(np.nan, index=raw.index)
        df = pd.DataFrame({
            "subj": raw[subj_col].astype(str),
            "block": raw[blk_col].astype(str) if blk_col else "1",
            "I": cong,
            "rt": rt,
            "correct": pd.to_numeric(raw[corr_col], errors="coerce"),
            "resp": resp_series.map(lambda x: str(x).strip() if pd.notna(x) else np.nan),
        })
        df = df.dropna(subset=["I", "rt", "correct"]).copy()
        df["correct"] = df["correct"].astype(int)
        df = df[(df["rt"] >= 0.15) & (df["rt"] <= 2.0)].reset_index(drop=True)
        cells[f"Hedge_Study2_Flanker_{session_tag}"] = add_conflict_trace(df, lam=lam)
    return cells


def read_eprime_table(path):
    for enc in ("utf-16", "utf-8-sig", "latin-1"):
        try:
            df = pd.read_csv(path, sep="\t", encoding=enc, quoting=csv.QUOTE_NONE, low_memory=False)
            if df.shape[1] >= 8 and len(df) >= 1000:
                df.columns = [str(c).strip().strip('"') for c in df.columns]
                return df
        except Exception:
            continue
    raise ValueError(f"Unable to decode E-Prime file {path}")


def load_mittelstaedt_exp2(data_dir, lam=0.5):
    node_dir = ensure_osf_node("nmy6b", data_dir)
    matches = list(node_dir.rglob("*Experiment2_SAT*.txt"))
    if not matches:
        raise FileNotFoundError(f"Missing Experiment2_SAT.txt under {node_dir}")
    raw = read_eprime_table(matches[0])

    rt1 = pd.to_numeric(raw.get("StimSlide1_RT"), errors="coerce")
    rt2 = pd.to_numeric(raw.get("StimSlide2_RT"), errors="coerce")
    acc1 = pd.to_numeric(raw.get("StimSlide1_ACC"), errors="coerce")
    acc2 = pd.to_numeric(raw.get("StimSlide2_ACC"), errors="coerce")
    use_s1 = rt1.notna() & (rt1 > 0)

    rt_ms = np.where(use_s1, rt1, rt2)
    acc = np.where(use_s1, acc1, acc2)
    if "StimSlide1_RESP" in raw.columns and "StimSlide2_RESP" in raw.columns:
        resp_raw = raw["StimSlide1_RESP"].where(use_s1, raw["StimSlide2_RESP"])
        resp = resp_raw.map(lambda x: str(x).strip().strip('"') if pd.notna(x) and str(x).strip() != "" else np.nan)
    else:
        resp = pd.Series(np.nan, index=raw.index)

    congru_num = pd.to_numeric(raw["Congru"], errors="coerce")
    # E-Prime Congru coding in Mittelstaedt Exp 2: 1 = congruent, 2 = incongruent
    cong = np.where(congru_num == 1, 1.0, np.where(congru_num == 2, -1.0, np.nan))

    task_col = "Task" if "Task" in raw.columns else "TaskType"
    sat_col = "SAT" if "SAT" in raw.columns else "Instruction"
    blk_col = "Block" if "Block" in raw.columns else "BlockList"

    base = pd.DataFrame({
        "subj": raw["Subject"].astype(str).str.strip().str.strip('"'),
        "block": raw[blk_col].astype(str).str.strip().str.strip('"'),
        "task": raw[task_col].astype(str).str.strip().str.strip('"'),
        "sat": raw[sat_col].astype(str).str.strip().str.strip('"'),
        "I": cong,
        "rt": rt_ms / 1000.0,
        "correct": acc,
        "resp": resp,
    })
    base = base.dropna(subset=["I", "rt", "correct"]).copy()
    base["correct"] = base["correct"].astype(int)
    base = base[(base["rt"] >= 0.15) & (base["rt"] <= 2.0)].reset_index(drop=True)

    cells = {}
    for (task_val, sat_val), grp in base.groupby(["task", "sat"], sort=True):
        cell_name = f"Mittelstaedt_Exp2_{task_val}_{sat_val}"
        cells[cell_name] = add_conflict_trace(grp.reset_index(drop=True), lam=lam)
    return cells


def vdw_g_norm(s_t, s_bar, gamma, delta):
    exp_bar = np.exp(np.clip(gamma * s_bar, -20.0, 20.0))
    exp_diff = np.exp(np.clip(-gamma * (s_t - s_bar), -20.0, 20.0))
    num = delta * exp_bar + (1.0 - delta) * exp_diff
    den = delta * exp_bar + (1.0 - delta)
    return num / np.maximum(den, 1e-12)


def fit_single_subject_m0(item):
    subj, grp = item
    rt = grp["rt"].to_numpy(dtype=float)
    ch = grp["correct"].to_numpy(dtype=int)
    cong = grp["I"].to_numpy(dtype=float)
    ter_max = max(0.06, float(np.min(rt)) - 0.015)

    res = minimize(
        lambda th: float(np.sum(wfpt_nll(rt, ch, th[0] + th[1] * cong, np.full_like(rt, th[2]), th[3]))),
        [2.5, 1.0, 1.1, min(0.22, ter_max * 0.7)],
        method="L-BFGS-B",
        bounds=[(-6.0, 8.0), (-5.0, 5.0), (0.3, 3.5), (0.04, ter_max)],
    )
    return subj, res.x


def fit_baselines(df, max_workers=4):
    groups = list(df.groupby("subj", sort=False))
    with ThreadPoolExecutor(max_workers=max_workers) as pool:
        return dict(pool.map(fit_single_subject_m0, groups))


def fit_cohort_models(df, baselines):
    sub = df[df["subj"].isin(baselines)].copy()
    pmat = np.vstack([baselines[s] for s in sub["subj"]])
    v0, c0, a0, ter = pmat.T
    rt = sub["rt"].to_numpy(dtype=float)
    ch = sub["correct"].to_numpy(dtype=int)
    cong = sub["I"].to_numpy(dtype=float)
    s_t = sub["S_t"].to_numpy(dtype=float)
    s_bar = sub["S_bar"].to_numpy(dtype=float)
    s_diff = s_t - s_bar
    log_n = np.log(len(sub))

    def eval_nll(gamma=0.0, eta=0.0, delta=0.0):
        g = vdw_g_norm(s_t, s_bar, gamma, delta) if (gamma != 0.0 or delta != 0.0) else 1.0
        v = v0 + c0 * g * cong
        a = np.clip(a0 * np.exp(np.clip(-eta * s_diff, -3.0, 3.0)), 0.2, 5.0) if eta != 0.0 else a0
        return float(np.sum(wfpt_nll(rt, ch, v, a, ter)))

    nll_m0 = eval_nll()

    r_m1 = minimize(lambda x: eval_nll(gamma=x[0]), [0.5], method="L-BFGS-B", bounds=[(-3.0, 5.0)])
    nll_m1 = min(nll_m0, float(r_m1.fun))
    g_m1 = float(r_m1.x[0]) if r_m1.fun < nll_m0 else 0.0

    r_m1b = minimize(lambda x: eval_nll(eta=x[0]), [-0.02], method="L-BFGS-B", bounds=[(-1.5, 1.5)])
    nll_m1b = min(nll_m0, float(r_m1b.fun))
    e_m1b = float(r_m1b.x[0]) if r_m1b.fun < nll_m0 else 0.0

    r_m1c = minimize(
        lambda x: eval_nll(gamma=x[0], eta=x[1]),
        [g_m1, e_m1b],
        method="L-BFGS-B",
        bounds=[(-3.0, 5.0), (-1.5, 1.5)],
    )
    if r_m1c.fun <= min(nll_m1, nll_m1b):
        nll_m1c, g_m1c, e_m1c = float(r_m1c.fun), float(r_m1c.x[0]), float(r_m1c.x[1])
    elif nll_m1 <= nll_m1b:
        nll_m1c, g_m1c, e_m1c = nll_m1, g_m1, 0.0
    else:
        nll_m1c, g_m1c, e_m1c = nll_m1b, 0.0, e_m1b

    best_m2 = (nll_m1, g_m1, 0.0)
    for d0 in (0.05, 0.25):
        r_m2 = minimize(
            lambda x: eval_nll(gamma=x[0], delta=x[1]),
            [max(g_m1, 0.1), d0],
            method="L-BFGS-B",
            bounds=[(0.0, 6.0), (0.0, 0.99)],
        )
        if r_m2.fun < best_m2[0]:
            best_m2 = (float(r_m2.fun), float(r_m2.x[0]), float(r_m2.x[1]))

    best_m3 = (nll_m1c, g_m1c, 0.0, e_m1c)
    for d0 in (0.05, 0.25):
        r_m3 = minimize(
            lambda x: eval_nll(gamma=x[0], delta=x[1], eta=x[2]),
            [max(g_m1c, 0.1), d0, e_m1c],
            method="L-BFGS-B",
            bounds=[(0.0, 6.0), (0.0, 0.99), (-1.5, 1.5)],
        )
        if r_m3.fun < best_m3[0]:
            best_m3 = (float(r_m3.fun), float(r_m3.x[0]), float(r_m3.x[1]), float(r_m3.x[2]))

    return {
        "n_subj": int(sub["subj"].nunique()),
        "n_trials": len(sub),
        "dbic_m1": 2.0 * (nll_m1 - nll_m0) + log_n,
        "dbic_m1b": 2.0 * (nll_m1b - nll_m0) + log_n,
        "dbic_m1c": 2.0 * (nll_m1c - nll_m0) + 2.0 * log_n,
        "dbic_m2": 2.0 * (best_m2[0] - nll_m0) + 2.0 * log_n,
        "dbic_m3": 2.0 * (best_m3[0] - nll_m0) + 3.0 * log_n,
        "dll_m1c_vs_m1": 2.0 * (nll_m1 - nll_m1c),
        "dll_m3_vs_m1c": 2.0 * (nll_m1c - best_m3[0]),
        "gamma": g_m1c,
        "eta": e_m1c,
        "delta": best_m3[2],
    }


def sim_cohort_ddm(tmpl_df, baselines, gamma, eta, delta, rng):
    pmat = np.vstack([baselines[s] for s in tmpl_df["subj"]])
    v0, c0, a0, ter = pmat.T
    cong = tmpl_df["I"].to_numpy(dtype=float)
    s_t = tmpl_df["S_t"].to_numpy(dtype=float)
    s_bar = tmpl_df["S_bar"].to_numpy(dtype=float)

    g = vdw_g_norm(s_t, s_bar, gamma, delta)
    v_t = v0 + c0 * g * cong
    a_t = np.clip(a0 * np.exp(np.clip(-eta * (s_t - s_bar), -3.0, 3.0)), 0.2, 5.0)

    n = len(tmpl_df)
    dt = 0.003
    max_steps = int(1.8 / dt)
    x = 0.5 * a_t
    alive = np.ones(n, dtype=bool)
    dt_hit = np.full(n, 1.8)
    correct = np.ones(n, dtype=int)
    sqrt_dt = np.sqrt(dt)

    for step in range(1, max_steps + 1):
        idx = np.flatnonzero(alive)
        if idx.size == 0:
            break
        x[idx] += v_t[idx] * dt + sqrt_dt * rng.standard_normal(idx.size)
        hi = idx[x[idx] >= a_t[idx]]
        lo = idx[x[idx] <= 0.0]
        if hi.size:
            dt_hit[hi] = step * dt
            correct[hi] = 1
            alive[hi] = False
        if lo.size:
            dt_hit[lo] = step * dt
            correct[lo] = 0
            alive[lo] = False

    rt = ter + dt_hit
    lapse = rng.random(n) < 0.005
    rt[lapse] = rng.uniform(0.15, 2.0, size=int(np.sum(lapse)))
    correct[lapse] = rng.integers(0, 2, size=int(np.sum(lapse)))

    sim = tmpl_df[["subj", "block", "I", "S_t", "S_bar"]].copy()
    sim["rt"] = np.clip(rt, 0.15, 2.0)
    sim["correct"] = correct
    return sim


def main():
    parser = argparse.ArgumentParser(description="SC-DDM robustness stress tests (Hedge & Mittelstaedt)")
    parser.add_argument("--data-dir", default=os.environ.get("SC_DDM_DATA_DIR", "./sc_ddm_workspace/data"))
    parser.add_argument("--out-dir", default=os.environ.get("SC_DDM_OUT_DIR", "./sc_ddm_workspace/stress_tests"))
    parser.add_argument("--reps", type=int, default=15)
    parser.add_argument("--workers", type=int, default=min(8, os.cpu_count() or 1))
    args, _ = parser.parse_known_args()

    out_dir = Path(args.out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    t0 = time.time()

    datasets = {}
    datasets.update(load_hedge_flanker(args.data_dir))
    datasets.update(load_mittelstaedt_exp2(args.data_dir))

    base_fits = {}
    base_rows = []
    for name, df in datasets.items():
        base_fits[name] = fit_baselines(df, max_workers=args.workers)
        res = fit_cohort_models(df, base_fits[name])
        base_rows.append({"cell": name, **res})
        print(f"baseline {name}: N={res['n_subj']}, trials={res['n_trials']}, M1c dBIC={res['dbic_m1c']:.2f}, M3 dBIC={res['dbic_m3']:.2f}, gamma={res['gamma']:.4f}, eta={res['eta']:.4f}, delta={res['delta']:.4f}")
    baseline_df = pd.DataFrame(base_rows)
    baseline_df.to_csv(out_dir / "baseline_fits.csv", index=False)

    fatigue_rows = []
    for name, df in datasets.items():
        flagged = add_flags(df)
        for label, mask in (
            ("early_half", flagged["trial_frac"] < 0.50),
            ("late_half", flagged["trial_frac"] >= 0.50),
            ("Q1_early", flagged["trial_frac"] < 0.25),
            ("Q4_late", flagged["trial_frac"] >= 0.75),
        ):
            res = fit_cohort_models(flagged.loc[mask].copy(), base_fits[name])
            fatigue_rows.append({"cell": name, "slice": label, **res})
    fatigue_df = pd.DataFrame(fatigue_rows)
    fatigue_df.to_csv(out_dir / "stress1_fatigue.csv", index=False)
    print(f"stress 1 complete ({time.time() - t0:.1f}s)")

    pes_rows = []
    for name, df in datasets.items():
        flagged = add_flags(df)
        subsets = [
            ("all_trials", np.ones(len(flagged), dtype=bool)),
            ("post_correct", flagged["prev_correct"] == 1),
        ]
        if flagged["resp_alt"].notna().any():
            subsets.append(("alt_only", flagged["resp_alt"] == 1))
            subsets.append(("post_correct_and_alt", (flagged["prev_correct"] == 1) & (flagged["resp_alt"] == 1)))
        for label, mask in subsets:
            res = fit_cohort_models(flagged.loc[mask].copy(), base_fits[name])
            pes_rows.append({"cell": name, "subset": label, **res})
    pes_df = pd.DataFrame(pes_rows)
    pes_df.to_csv(out_dir / "stress2_pes_priming.csv", index=False)
    print(f"stress 2 complete ({time.time() - t0:.1f}s)")

    lam_rows = []
    for name in ("Hedge_Study2_Flanker_S1", "Mittelstaedt_Exp2_Flanker_Speed"):
        raw_df = datasets[name]
        for lam in (0.2, 0.4, 0.6, 0.8, 1.0):
            traced = add_conflict_trace(raw_df, lam=lam)
            res = fit_cohort_models(traced, base_fits[name])
            lam_rows.append({"cell": name, "lambda": lam, **res})
    lam_df = pd.DataFrame(lam_rows)
    lam_df.to_csv(out_dir / "stress3_lambda_grid.csv", index=False)
    print(f"stress 3 complete ({time.time() - t0:.1f}s)")

    rng = np.random.default_rng(123)
    power_rows = []
    cal_specs = [
        ("Hedge_Study2_Flanker_S1", 0.415, -0.0065),
        ("Mittelstaedt_Exp2_Flanker_Speed", 0.775, -0.0274),
    ]
    for name, g_true, e_true in cal_specs:
        tmpl_df = datasets[name]
        b_fits = base_fits[name]
        for d_true in (0.0, 0.25, 0.50, 0.70, 0.85):
            wins = 0
            d_hats = []
            gains = []
            for _ in range(args.reps):
                sim_df = sim_cohort_ddm(tmpl_df, b_fits, gamma=g_true, eta=e_true, delta=d_true, rng=rng)
                res = fit_cohort_models(sim_df, b_fits)
                gain = min(res["dbic_m2"], res["dbic_m3"]) - res["dbic_m1c"]
                if gain < -2.0:
                    wins += 1
                d_hats.append(res["delta"])
                gains.append(gain)
            power_rows.append({
                "cell": name,
                "delta_true": d_true,
                "n_reps": args.reps,
                "power_dbic_lt_neg2": wins / args.reps,
                "mean_delta_hat": float(np.mean(d_hats)),
                "median_vdw_minus_m1c_dbic": float(np.median(gains)),
            })
    power_df = pd.DataFrame(power_rows)
    power_df.to_csv(out_dir / "stress4_delta_power.csv", index=False)
    print(f"stress 4 complete; all outputs saved to {out_dir.resolve()} ({time.time() - t0:.1f}s)")

    return {
        "OUT_DIR": str(out_dir),
        "datasets": datasets,
        "base_fits": base_fits,
        "baseline_df": baseline_df,
        "fatigue_df": fatigue_df,
        "pes_df": pes_df,
        "lam_df": lam_df,
        "power_df": power_df,
    }


if __name__ == "__main__":
    _state = main()
    OUT_DIR = _state["OUT_DIR"]
    datasets = _state["datasets"]
    base_fits = _state["base_fits"]
    baseline_df = _state["baseline_df"]
    fatigue_df = _state["fatigue_df"]
    pes_df = _state["pes_df"]
    lam_df = _state["lam_df"]
    power_df = _state["power_df"]

KeyboardInterrupt: 

In [ ]:
import numpy as np
import pandas as pd
from scipy.optimize import minimize

_K_SMALL = np.arange(-6, 7, dtype=float)
_K_LARGE = np.arange(1, 18, 2, dtype=float)
_SIN_LARGE = np.where((_K_LARGE.astype(int) // 2) % 2 == 0, 1.0, -1.0)


def wfpt_pdf(rt, choice, v, a, ter=0.0):
    rt = np.asarray(rt, dtype=float)
    choice = np.broadcast_to(np.asarray(choice, dtype=int), rt.shape)
    v = np.broadcast_to(np.asarray(v, dtype=float), rt.shape)
    a = np.broadcast_to(np.asarray(a, dtype=float), rt.shape)
    ter = np.broadcast_to(np.asarray(ter, dtype=float), rt.shape)

    t = rt - ter
    out = np.zeros_like(rt, dtype=float)
    valid = (t > 1e-6) & (a > 1e-4)
    if not np.any(valid):
        return out

    tv = t[valid]
    vv = v[valid]
    av = a[valid]
    sign = np.where(choice[valid] == 1, 1.0, -1.0)
    tt = tv / (av * av)

    # Navarro & Fuss (2009) small-time / large-time series at unbiased start z = a/2
    f_star = np.empty_like(tt)
    small = tt < 0.25
    if np.any(small):
        ts = tt[small][:, None]
        w_term = 0.5 + 2.0 * _K_SMALL[None, :]
        s_sum = np.sum(w_term * np.exp(-(w_term * w_term) / (2.0 * ts)), axis=1)
        f_star[small] = s_sum / np.sqrt(2.0 * np.pi * (tt[small] ** 3))

    large = ~small
    if np.any(large):
        tl = tt[large][:, None]
        l_sum = np.sum(
            _K_LARGE[None, :] * _SIN_LARGE[None, :] * np.exp(-0.5 * (_K_LARGE[None, :] ** 2) * (np.pi ** 2) * tl),
            axis=1,
        )
        f_star[large] = np.pi * l_sum

    f_star = np.maximum(f_star, 0.0)
    log_pref = sign * 0.5 * vv * av - 0.5 * (vv * vv) * tv - 2.0 * np.log(av)
    out[valid] = np.exp(np.clip(log_pref, -700.0, 50.0)) * f_star
    return out


def wfpt_nll(rt, choice, v, a, ter, rt_span=2.0, p_lapse=0.005):
    pdf = wfpt_pdf(rt, choice, v, a, ter)
    mix = (1.0 - p_lapse) * pdf + p_lapse / rt_span
    return -np.log(np.maximum(mix, 1e-300))


def add_conflict_trace(df, lam=0.5):
    out = df.copy()
    traces = np.zeros(len(out), dtype=float)
    group_cols = ["subj", "block"] if "block" in out.columns and out["block"].notna().any() else ["subj"]
    for _, idx in out.groupby(group_cols, sort=False).groups.items():
        cong = out.loc[idx, "I"].to_numpy(dtype=float)
        s = np.zeros_like(cong)
        for t in range(1, len(cong)):
            s[t] = lam * s[t - 1] + 0.5 * (1.0 - cong[t - 1])
        traces[idx] = s
    out["S_t"] = traces
    out["S_bar"] = out.groupby("subj")["S_t"].transform("mean")
    return out


def vdw_g_norm(s_t, s_bar, gamma, delta):
    exp_bar = np.exp(np.clip(gamma * s_bar, -20.0, 20.0))
    exp_diff = np.exp(np.clip(-gamma * (s_t - s_bar), -20.0, 20.0))
    num = delta * exp_bar + (1.0 - delta) * exp_diff
    den = delta * exp_bar + (1.0 - delta)
    return num / np.maximum(den, 1e-12)


def fit_m0_baselines(df):
    baselines = {}
    for subj, grp in df.groupby("subj", sort=False):
        rt = grp["rt"].to_numpy(dtype=float)
        ch = grp["correct"].to_numpy(dtype=int)
        cong = grp["I"].to_numpy(dtype=float)
        ter_max = max(0.06, float(np.min(rt)) - 0.015)

        def nll(theta):
            v0, c0, a0, ter = theta
            v = v0 + c0 * cong
            a = np.full_like(rt, a0)
            return float(np.sum(wfpt_nll(rt, ch, v, a, ter)))

        x0 = np.array([2.0, 0.5, 1.0, min(0.25, ter_max * 0.7)])
        bounds = [(-6.0, 8.0), (-5.0, 5.0), (0.3, 3.5), (0.04, ter_max)]
        res = minimize(nll, x0, method="L-BFGS-B", bounds=bounds)
        baselines[subj] = res.x
    return baselines


def fit_cohort_models(df, baselines=None):
    if baselines is None:
        baselines = fit_m0_baselines(df)

    sub_df = df[df["subj"].isin(baselines)].copy()
    rt = sub_df["rt"].to_numpy(dtype=float)
    ch = sub_df["correct"].to_numpy(dtype=int)
    cong = sub_df["I"].to_numpy(dtype=float)
    s_t = sub_df["S_t"].to_numpy(dtype=float)
    s_bar = sub_df["S_bar"].to_numpy(dtype=float)
    s_diff = s_t - s_bar

    pmat = np.vstack([baselines[s] for s in sub_df["subj"]])
    v0, c0, a0, ter = pmat[:, 0], pmat[:, 1], pmat[:, 2], pmat[:, 3]
    n_trials = len(sub_df)
    log_n = np.log(max(n_trials, 1))

    def eval_nll(gamma=0.0, eta=0.0, delta=0.0):
        g = vdw_g_norm(s_t, s_bar, gamma, delta) if (gamma != 0.0 or delta != 0.0) else 1.0
        v = v0 + c0 * g * cong
        a = a0 * np.exp(np.clip(-eta * s_diff, -3.0, 3.0)) if eta != 0.0 else a0
        a = np.clip(a, 0.2, 5.0)
        return float(np.sum(wfpt_nll(rt, ch, v, a, ter)))

    nll_m0 = eval_nll()

    r_m1 = minimize(lambda x: eval_nll(gamma=x[0]), [0.5], method="L-BFGS-B", bounds=[(-3.0, 5.0)])
    nll_m1 = min(nll_m0, float(r_m1.fun))
    g_m1 = float(r_m1.x[0]) if r_m1.fun < nll_m0 else 0.0

    r_m1b = minimize(lambda x: eval_nll(eta=x[0]), [-0.03], method="L-BFGS-B", bounds=[(-1.5, 1.5)])
    nll_m1b = min(nll_m0, float(r_m1b.fun))
    e_m1b = float(r_m1b.x[0]) if r_m1b.fun < nll_m0 else 0.0

    r_m1c = minimize(
        lambda x: eval_nll(gamma=x[0], eta=x[1]),
        [g_m1, e_m1b],
        method="L-BFGS-B",
        bounds=[(-3.0, 5.0), (-1.5, 1.5)],
    )
    if r_m1c.fun <= min(nll_m1, nll_m1b):
        nll_m1c, g_m1c, e_m1c = float(r_m1c.fun), float(r_m1c.x[0]), float(r_m1c.x[1])
    elif nll_m1 <= nll_m1b:
        nll_m1c, g_m1c, e_m1c = nll_m1, g_m1, 0.0
    else:
        nll_m1c, g_m1c, e_m1c = nll_m1b, 0.0, e_m1b

    best_m2 = (nll_m1, g_m1, 0.0)
    for d0 in (0.05, 0.25):
        r_m2 = minimize(
            lambda x: eval_nll(gamma=x[0], delta=x[1]),
            [max(g_m1, 0.1), d0],
            method="L-BFGS-B",
            bounds=[(0.0, 6.0), (0.0, 0.99)],
        )
        if r_m2.fun < best_m2[0]:
            best_m2 = (float(r_m2.fun), float(r_m2.x[0]), float(r_m2.x[1]))

    best_m3 = (nll_m1c, g_m1c, 0.0, e_m1c)
    for d0 in (0.05, 0.25):
        r_m3 = minimize(
            lambda x: eval_nll(gamma=x[0], delta=x[1], eta=x[2]),
            [max(g_m1c, 0.1), d0, e_m1c],
            method="L-BFGS-B",
            bounds=[(0.0, 6.0), (0.0, 0.99), (-1.5, 1.5)],
        )
        if r_m3.fun < best_m3[0]:
            best_m3 = (float(r_m3.fun), float(r_m3.x[0]), float(r_m3.x[1]), float(r_m3.x[2]))

    return {
        "n_subj": int(sub_df["subj"].nunique()),
        "n_trials": n_trials,
        "dbic_m1": 2.0 * (nll_m1 - nll_m0) + 1.0 * log_n,
        "dbic_m1b": 2.0 * (nll_m1b - nll_m0) + 1.0 * log_n,
        "dbic_m1c": 2.0 * (nll_m1c - nll_m0) + 2.0 * log_n,
        "dbic_m2": 2.0 * (best_m2[0] - nll_m0) + 2.0 * log_n,
        "dbic_m3": 2.0 * (best_m3[0] - nll_m0) + 3.0 * log_n,
        "dll_m1c_vs_m1": 2.0 * (nll_m1 - nll_m1c),
        "dll_m3_vs_m1c": 2.0 * (nll_m1c - best_m3[0]),
        "gamma": g_m1c,
        "eta": e_m1c,
        "delta": best_m3[2],
    }


trapezoid_fn = getattr(np, "trapezoid", getattr(np, "trapz", None))
grid = np.linspace(0.001, 6.0, 6000)
mass = trapezoid_fn(wfpt_pdf(grid, 1, 1.5, 1.1) + wfpt_pdf(grid, 0, 1.5, 1.1), grid)
print(f"wfpt integral check: {mass:.6f}")

wfpt integral check: 1.000000


In [ ]:
def load_mittelstaedt(experiments=("experiment2",)):
    files = osf_walk("https://api.osf.io/v2/nodes/nmy6b/files/osfstorage/")
    cells = {}
    for exp_tag in experiments:
        matches = [
            f for f in files
            if exp_tag in os.path.basename(f[0]).lower()
            and f[0].lower().endswith((".txt", ".csv", ".tsv"))
            and not any(bad in f[0].lower() for bad in ("readme", "info", "codebook", "desc"))
        ]
        if not matches:
            continue
        matches.sort(key=lambda x: ("sat" not in os.path.basename(x[0]).lower(), len(x[0])))
        fpath, url = matches[0]
        local_path = download_file(url, os.path.join(DATA_DIR, "nmy6b", os.path.basename(fpath)))
        raw = read_table(local_path)
        cols = {c.lower(): c for c in raw.columns}

        subj_col = pick_col(cols, "subno", "vp", "subj", "subject", "participant")
        comp_col = pick_col(cols, "congru", "comp", "cong")
        task_col = pick_col(cols, "task")
        sat_col = pick_col(cols, "cond", "sat", "instruction", "speed")
        block_col = pick_col(cols, "blkno", "block", exclude=("list", "trial", "total", "errors", "correct", "mean"))
        proc_col = pick_col(cols, "procedure_trial_", "running_trial_", "procedure")

        trial_mask = raw[comp_col].notna()
        if proc_col is not None:
            trial_mask = trial_mask & ~raw[proc_col].astype(str).str.lower().str.contains("prac")

        sub = raw.loc[trial_mask].copy()

        # Pair each E-Prime StimSlide*_RT with its matching StimSlide*_ACC
        slide_rt_cols = [c for c in sub.columns if re.fullmatch(r"stimslide\d*_rt", c.lower())]
        slide_acc_cols = [c for c in sub.columns if re.fullmatch(r"stimslide\d*_acc", c.lower())]
        if slide_rt_cols and slide_acc_cols:
            rt_series = sub[slide_rt_cols].apply(pd.to_numeric, errors="coerce").bfill(axis=1).iloc[:, 0]
            acc_series = sub[slide_acc_cols].apply(pd.to_numeric, errors="coerce").bfill(axis=1).iloc[:, 0]
        else:
            rt_series, slide_rt_cols = coalesce_trial_cols(
                sub, ("rt", "_rt", ".rt"), exclude=("includeforrt", "blockmean", "fixslide", "prac", "onset", "duration", "feedback")
            )
            acc_series, slide_acc_cols = coalesce_trial_cols(
                sub, ("_acc", ".acc"), exclude=("blockmean", "fixslide", "prac", "onset", "duration", "feedback", "cresp")
            )
        resp_series, _ = coalesce_trial_cols(sub, ("_resp", ".resp", "resp"))
        print(f"nmy6b paired cols -> rt={slide_rt_cols}, acc={slide_acc_cols}")

        df = pd.DataFrame()
        df["subj"] = sub[subj_col].astype(str).str.strip('"')
        df["rt"] = rt_series / 1000.0 if rt_series.median() > 10 else rt_series
        df["correct"] = np.where(acc_series.notna(), (acc_series > 0).astype(float), np.nan)

        comp_str = sub[comp_col].astype(str).str.strip().str.strip('"').str.lower()
        df["I"] = np.where(comp_str.str.contains("incomp|incong|^0$|^-1$"), -1.0, 1.0)
        df["block"] = sub[block_col] if block_col else 1
        df["resp"] = resp_series.astype(str) if resp_series is not None else np.nan
        df["task"] = sub[task_col].astype(str).str.strip().str.strip('"').str.title() if task_col else "Task"
        df["sat"] = sub[sat_col].astype(str).str.strip().str.strip('"').str.title() if sat_col else "All"

        df = df.dropna(subset=["rt", "correct", "I"]).copy()
        df["correct"] = df["correct"].astype(int)
        df = df[(df["rt"] >= 0.15) & (df["rt"] <= 2.0)].reset_index(drop=True)
        exp_label = exp_tag.replace("experiment", "Exp")
        for (task, sat), grp in df.groupby(["task", "sat"]):
            if len(grp) > 500:
                cells[f"Mittelstaedt_{exp_label}_{task}_{sat}"] = add_conflict_trace(grp.reset_index(drop=True), lam=0.5)
    print(f"mittelstaedt loaded: {len(cells)} cells")
    return cells


datasets = {}
datasets.update(load_mittelstaedt(experiments=("experiment2",)))
datasets.update(load_hedge_flanker())

for name, df in datasets.items():
    print(f"{name}: N={df['subj'].nunique()}, trials={len(df)}, acc={df['correct'].mean():.3f}, med_rt={df['rt'].median():.3f}")

nmy6b paired cols -> rt=['StimSlide_RT', 'StimSlide1_RT', 'StimSlide2_RT', 'StimSlide3_RT'], acc=['StimSlide_ACC', 'StimSlide1_ACC', 'StimSlide2_ACC', 'StimSlide3_ACC']
mittelstaedt loaded: 4 cells
hedge loaded: 2 cells
Mittelstaedt_Exp2_1_1: N=32, trials=7117, acc=0.851, med_rt=0.373
Mittelstaedt_Exp2_1_2: N=32, trials=7166, acc=0.962, med_rt=0.438
Mittelstaedt_Exp2_2_1: N=32, trials=7107, acc=0.837, med_rt=0.383
Mittelstaedt_Exp2_2_2: N=32, trials=7158, acc=0.959, med_rt=0.454
Hedge_Study2_Flanker_S1: N=60, trials=42192, acc=0.922, med_rt=0.418
Hedge_Study2_Flanker_S2: N=60, trials=41083, acc=0.897, med_rt=0.405


In [ ]:
base_rows = []
base_fits = {}

for name, df in datasets.items():
    baselines = fit_m0_baselines(df)
    base_fits[name] = baselines
    res = fit_cohort_models(df, baselines=baselines)
    base_rows.append({"cell": name, **res})
    print(
        f"{name}: N={res['n_subj']}, trials={res['n_trials']}, "
        f"M1c dBIC={res['dbic_m1c']:.2f}, M3 dBIC={res['dbic_m3']:.2f}, "
        f"gamma={res['gamma']:.3f}, eta={res['eta']:.4f}, delta={res['delta']:.3f}"
    )

baseline_df = pd.DataFrame(base_rows)
baseline_df.to_csv(f"{OUT_DIR}/baseline_check.csv", index=False)

Mittelstaedt_Exp2_1_1: N=32, trials=7117, M1c dBIC=17.74, M3 dBIC=26.61, gamma=0.000, eta=0.0000, delta=0.000
Mittelstaedt_Exp2_1_2: N=32, trials=7166, M1c dBIC=17.75, M3 dBIC=26.63, gamma=0.000, eta=0.0000, delta=0.000
Mittelstaedt_Exp2_2_1: N=32, trials=7107, M1c dBIC=17.74, M3 dBIC=26.61, gamma=0.000, eta=0.0000, delta=0.000
Mittelstaedt_Exp2_2_2: N=32, trials=7158, M1c dBIC=17.75, M3 dBIC=26.63, gamma=0.000, eta=0.0000, delta=0.000
Hedge_Study2_Flanker_S1: N=60, trials=42192, M1c dBIC=-78.45, M3 dBIC=-67.80, gamma=0.415, eta=-0.0065, delta=0.000
Hedge_Study2_Flanker_S2: N=60, trials=41083, M1c dBIC=-37.19, M3 dBIC=-26.57, gamma=0.308, eta=-0.0128, delta=0.000


In [ ]:
def add_flags(df):
    out = df.copy()
    counts = out.groupby("subj")["rt"].transform("count")
    out["trial_frac"] = out.groupby("subj").cumcount() / counts
    out["prev_correct"] = out.groupby("subj")["correct"].shift(1)
    if "resp" in out.columns and out["resp"].notna().any():
        out["resp_alt"] = out.groupby("subj")["resp"].shift(1) != out["resp"]
    else:
        out["resp_alt"] = np.nan
    return out


fatigue_rows = []
for name, df in datasets.items():
    flagged = add_flags(df)
    slices = [
        ("early_half", flagged["trial_frac"] < 0.50),
        ("late_half", flagged["trial_frac"] >= 0.50),
        ("Q1_early", flagged["trial_frac"] < 0.25),
        ("Q4_late", flagged["trial_frac"] >= 0.75),
    ]
    for label, mask in slices:
        sub = flagged.loc[mask].copy()
        res = fit_cohort_models(sub, baselines=base_fits[name])
        fatigue_rows.append({"cell": name, "slice": label, **res})

fatigue_df = pd.DataFrame(fatigue_rows)
fatigue_df.to_csv(f"{OUT_DIR}/stress1_fatigue.csv", index=False)
print("stress 1: fatigue split")
print(fatigue_df[["cell", "slice", "n_trials", "dbic_m1c", "dbic_m3", "gamma", "eta", "delta"]].to_string(index=False))

pes_rows = []
for name, df in datasets.items():
    flagged = add_flags(df)
    subsets = [
        ("all_trials", np.ones(len(flagged), dtype=bool)),
        ("post_correct", flagged["prev_correct"] == 1),
    ]
    if flagged["resp_alt"].notna().any():
        subsets.append(("alt_only", flagged["resp_alt"] == True))
        subsets.append(("post_correct_and_alt", (flagged["prev_correct"] == 1) & (flagged["resp_alt"] == True)))
    for label, mask in subsets:
        sub = flagged.loc[mask].copy()
        res = fit_cohort_models(sub, baselines=base_fits[name])
        pes_rows.append({"cell": name, "subset": label, **res})

pes_df = pd.DataFrame(pes_rows)
pes_df.to_csv(f"{OUT_DIR}/stress2_pes_priming.csv", index=False)
print("\nstress 2: post-error and alternation controls")
print(pes_df[["cell", "subset", "n_trials", "dbic_m1c", "dbic_m3", "dll_m1c_vs_m1", "gamma", "eta", "delta"]].to_string(index=False))

flagship_cells = [c for c in datasets if any(k in c for k in ("Simon_Speed", "Flanker_S1", "PACT_"))][:3]
lam_rows = []
for name in flagship_cells:
    for lam in (0.2, 0.4, 0.6, 0.8, 1.0):
        df_lam = add_conflict_trace(datasets[name], lam=lam)
        res = fit_cohort_models(df_lam, baselines=base_fits[name])
        lam_rows.append({"cell": name, "lambda": lam, **res})

lam_df = pd.DataFrame(lam_rows)
lam_df.to_csv(f"{OUT_DIR}/stress3_lambda_grid.csv", index=False)
print("\nstress 3: lambda grid")
print(lam_df[["cell", "lambda", "dbic_m1", "dbic_m1c", "dbic_m2", "dbic_m3", "gamma", "eta", "delta"]].to_string(index=False))


def sim_cohort_ddm(template_df, baselines, gamma, eta, delta, rng, dt=0.002, p_lapse=0.005):
    sub = template_df[template_df["subj"].isin(baselines)].copy().reset_index(drop=True)
    pmat = np.vstack([baselines[s] for s in sub["subj"]])
    v0, c0, a0, ter = pmat[:, 0], pmat[:, 1], pmat[:, 2], pmat[:, 3]
    cong = sub["I"].to_numpy(dtype=float)
    s_t = sub["S_t"].to_numpy(dtype=float)
    s_bar = sub["S_bar"].to_numpy(dtype=float)

    g = vdw_g_norm(s_t, s_bar, gamma, delta)
    v = v0 + c0 * g * cong
    a = np.clip(a0 * np.exp(np.clip(-eta * (s_t - s_bar), -3.0, 3.0)), 0.2, 5.0)

    n = len(sub)
    x = 0.5 * a
    alive = np.ones(n, dtype=bool)
    dt_hit = np.full(n, 1.8)
    correct = np.ones(n, dtype=int)
    sqrt_dt = np.sqrt(dt)
    max_steps = int(1.8 / dt)

    for step in range(1, max_steps + 1):
        idx = np.flatnonzero(alive)
        if idx.size == 0:
            break
        x[idx] += v[idx] * dt + sqrt_dt * rng.standard_normal(idx.size)
        hi = idx[x[idx] >= a[idx]]
        lo = idx[x[idx] <= 0.0]
        if hi.size:
            dt_hit[hi] = step * dt
            correct[hi] = 1
            alive[hi] = False
        if lo.size:
            dt_hit[lo] = step * dt
            correct[lo] = 0
            alive[lo] = False

    rt = ter + dt_hit
    lapse = rng.random(n) < p_lapse
    if np.any(lapse):
        rt[lapse] = rng.uniform(0.15, 2.0, size=int(np.sum(lapse)))
        correct[lapse] = rng.integers(0, 2, size=int(np.sum(lapse)))

    out = sub[["subj", "block", "I", "S_t", "S_bar"]].copy()
    out["rt"] = np.clip(rt, 0.15, 2.0)
    out["correct"] = correct
    return out


power_targets = [c for c in ["Hedge_Study2_Flanker_S1", "Mittelstaedt_Exp2_1_1"] if c in datasets]
rng = np.random.default_rng(42)
power_rows = []
n_reps = 15

for power_target in power_targets:
    base_row = baseline_df.loc[baseline_df["cell"] == power_target].iloc[0]
    for d_true in (0.0, 0.05, 0.10, 0.15, 0.20, 0.30, 0.40):
        wins = 0
        d_hats = []
        gains = []
        for _ in range(n_reps):
            sim_df = sim_cohort_ddm(
                datasets[power_target],
                base_fits[power_target],
                gamma=float(base_row["gamma"]),
                eta=float(base_row["eta"]),
                delta=d_true,
                rng=rng,
            )
            res = fit_cohort_models(sim_df, baselines=base_fits[power_target])
            gain = min(res["dbic_m2"], res["dbic_m3"]) - res["dbic_m1c"]
            if gain < -2.0:
                wins += 1
            d_hats.append(res["delta"])
            gains.append(gain)
        power_rows.append({
            "cell": power_target,
            "delta_true": d_true,
            "n_reps": n_reps,
            "power_dbic_lt_neg2": wins / n_reps,
            "mean_delta_hat": float(np.mean(d_hats)),
            "median_vdw_minus_m1c_dbic": float(np.median(gains)),
        })
        print(f"{power_target} | delta={d_true:.2f} | power={wins}/{n_reps} | mean_hat={np.mean(d_hats):.3f}")

power_df = pd.DataFrame(power_rows)
power_df.to_csv(f"{OUT_DIR}/stress4_delta_power.csv", index=False)
print("\nstress 4: injected-delta power")
print(power_df.to_string(index=False))

stress 1: fatigue split
                   cell      slice  n_trials   dbic_m1c    dbic_m3    gamma       eta    delta
  Mittelstaedt_Exp2_1_1 early_half      3560  16.355032  24.532547 0.000000  0.000000 0.000000
  Mittelstaedt_Exp2_1_1  late_half      3557  16.353346  24.530018 0.000000  0.000000 0.000000
  Mittelstaedt_Exp2_1_1   Q1_early      1781  14.969861  22.454791 0.000000  0.000000 0.000000
  Mittelstaedt_Exp2_1_1    Q4_late      1777  14.965364  22.448045 0.000000  0.000000 0.000000
  Mittelstaedt_Exp2_1_2 early_half      3583  16.367911  24.551867 0.000000  0.000000 0.000000
  Mittelstaedt_Exp2_1_2  late_half      3583  16.367911  24.551867 0.000000  0.000000 0.000000
  Mittelstaedt_Exp2_1_2   Q1_early      1792  14.982175  22.473263 0.000000  0.000000 0.000000
  Mittelstaedt_Exp2_1_2    Q4_late      1791  14.981059  22.471588 0.000000  0.000000 0.000000
  Mittelstaedt_Exp2_2_1 early_half      3556  16.352783  24.529175 0.000000  0.000000 0.000000
  Mittelstaedt_Exp2_2_1  l

In [ ]:
def load_mittelstaedt_fixed():
    local_path = os.path.join(DATA_DIR, "nmy6b", "Experiment2_SAT.txt")
    raw = read_table(local_path)
    cols = {c.lower(): c for c in raw.columns}

    subj_col = pick_col(cols, "subno", "vp", "subj", "subject", "participant")
    comp_col = pick_col(cols, "congru", "comp", "cong")
    task_col = pick_col(cols, "task")
    sat_col = pick_col(cols, "cond", "sat", "instruction", "speed")
    block_col = pick_col(cols, "blkno", "block", exclude=("list", "trial", "total", "errors", "correct", "mean"))
    proc_col = pick_col(cols, "procedure_trial_", "running_trial_", "procedure")

    trial_mask = raw[comp_col].notna()
    if proc_col is not None:
        trial_mask = trial_mask & ~raw[proc_col].astype(str).str.lower().str.contains("prac")

    sub = raw.loc[trial_mask].copy()
    slide_rt_cols = [c for c in sub.columns if re.fullmatch(r"stimslide\d*_rt", c.lower())]
    slide_acc_cols = [c for c in sub.columns if re.fullmatch(r"stimslide\d*_acc", c.lower())]
    rt_series = sub[slide_rt_cols].apply(pd.to_numeric, errors="coerce").bfill(axis=1).iloc[:, 0]
    acc_series = sub[slide_acc_cols].apply(pd.to_numeric, errors="coerce").bfill(axis=1).iloc[:, 0]
    resp_series, _ = coalesce_trial_cols(sub, ("_resp", ".resp", "resp"))

    df = pd.DataFrame()
    df["subj"] = sub[subj_col].astype(str).str.strip('"')
    df["rt"] = rt_series / 1000.0 if rt_series.median() > 10 else rt_series
    df["correct"] = np.where(acc_series.notna(), (acc_series > 0).astype(float), np.nan)

    # E-Prime Congru codes congruent=1, incongruent=2
    comp_num = pd.to_numeric(sub[comp_col], errors="coerce")
    if set(comp_num.dropna().unique()) <= {1.0, 2.0}:
        df["I"] = np.where(comp_num == 2.0, -1.0, 1.0)
    else:
        comp_str = sub[comp_col].astype(str).str.strip().str.strip('"').str.lower()
        df["I"] = np.where(comp_str.str.contains("incomp|incong|^2$|^0$|^-1$"), -1.0, 1.0)

    df["block"] = sub[block_col] if block_col else 1
    df["resp"] = resp_series.astype(str) if resp_series is not None else np.nan
    task_map = {"1": "Flanker", "2": "Simon"}
    sat_map = {"1": "Speed", "2": "Accuracy"}
    df["task"] = sub[task_col].astype(str).str.strip().str.strip('"').map(lambda x: task_map.get(x, x.title()))
    df["sat"] = sub[sat_col].astype(str).str.strip().str.strip('"').map(lambda x: sat_map.get(x, x.title()))

    df = df.dropna(subset=["rt", "correct", "I"]).copy()
    df["correct"] = df["correct"].astype(int)
    df = df[(df["rt"] >= 0.15) & (df["rt"] <= 2.0)].reset_index(drop=True)

    cells = {}
    for (task, sat), grp in df.groupby(["task", "sat"]):
        if len(grp) > 500:
            cells[f"Mittelstaedt_Exp2_{task}_{sat}"] = add_conflict_trace(grp.reset_index(drop=True), lam=0.5)
    return cells


mitt_cells = load_mittelstaedt_fixed()
mitt_fits = {}
mitt_rows = []
for name, df in mitt_cells.items():
    b = fit_m0_baselines(df)
    mitt_fits[name] = b
    res = fit_cohort_models(df, baselines=b)
    mitt_rows.append({"cell": name, **res})
    print(f"{name}: N={res['n_subj']}, trials={res['n_trials']}, M1c dBIC={res['dbic_m1c']:.2f}, M3 dBIC={res['dbic_m3']:.2f}, gamma={res['gamma']:.3f}, eta={res['eta']:.4f}, delta={res['delta']:.3f}")

pd.DataFrame(mitt_rows).to_csv(f"{OUT_DIR}/mittelstaedt_baseline.csv", index=False)

m_lam = []
for name in [c for c in mitt_cells if "Speed" in c]:
    for lam in (0.2, 0.4, 0.6, 0.8, 1.0):
        res = fit_cohort_models(add_conflict_trace(mitt_cells[name], lam=lam), baselines=mitt_fits[name])
        m_lam.append({"cell": name, "lambda": lam, **res})
print("\nmittelstaedt lambda grid:")
print(pd.DataFrame(m_lam)[["cell", "lambda", "dbic_m1", "dbic_m1c", "dbic_m2", "dbic_m3", "gamma", "eta", "delta"]].to_string(index=False))

Mittelstaedt_Exp2_Flanker_Accuracy: N=32, trials=7166, M1c dBIC=-59.20, M3 dBIC=-50.32, gamma=0.843, eta=-0.0115, delta=0.000
Mittelstaedt_Exp2_Flanker_Speed: N=32, trials=7117, M1c dBIC=-111.57, M3 dBIC=-102.70, gamma=0.775, eta=-0.0274, delta=0.000
Mittelstaedt_Exp2_Simon_Accuracy: N=32, trials=7158, M1c dBIC=-21.32, M3 dBIC=-12.74, gamma=0.436, eta=-0.0148, delta=0.272
Mittelstaedt_Exp2_Simon_Speed: N=32, trials=7107, M1c dBIC=0.32, M3 dBIC=9.08, gamma=0.265, eta=0.0037, delta=0.327

mittelstaedt lambda grid:
                           cell  lambda     dbic_m1    dbic_m1c     dbic_m2     dbic_m3    gamma       eta    delta
Mittelstaedt_Exp2_Flanker_Speed     0.2 -128.097679 -129.987114 -119.227438 -121.116872 1.015838 -0.034044 0.000000
Mittelstaedt_Exp2_Flanker_Speed     0.4 -122.042547 -123.412243 -113.172305 -114.542002 0.883435 -0.030887 0.000000
Mittelstaedt_Exp2_Flanker_Speed     0.6  -94.403557  -92.742652  -85.533315  -83.872411 0.638456 -0.022257 0.000000
Mittelstaedt_Exp2_

In [ ]:
m_fatigue = []
for name, df in mitt_cells.items():
    flagged = add_flags(df)
    for label, mask in [
        ("early_half", flagged["trial_frac"] < 0.50),
        ("late_half", flagged["trial_frac"] >= 0.50),
        ("Q1_early", flagged["trial_frac"] < 0.25),
        ("Q4_late", flagged["trial_frac"] >= 0.75),
    ]:
        sub = flagged.loc[mask].copy()
        res = fit_cohort_models(sub, baselines=mitt_fits[name])
        m_fatigue.append({"cell": name, "slice": label, **res})

m_fatigue_df = pd.DataFrame(m_fatigue)
m_fatigue_df.to_csv(f"{OUT_DIR}/mittelstaedt_stress1_fatigue.csv", index=False)
print("mittelstaedt stress 1: fatigue split")
print(m_fatigue_df[["cell", "slice", "n_trials", "dbic_m1c", "dbic_m3", "gamma", "eta", "delta"]].to_string(index=False))

m_pes = []
for name, df in mitt_cells.items():
    flagged = add_flags(df)
    subsets = [
        ("all_trials", np.ones(len(flagged), dtype=bool)),
        ("post_correct", flagged["prev_correct"] == 1),
    ]
    if flagged["resp_alt"].notna().any():
        subsets.append(("alt_only", flagged["resp_alt"] == 1))
        subsets.append(("post_correct_and_alt", (flagged["prev_correct"] == 1) & (flagged["resp_alt"] == 1)))
    for label, mask in subsets:
        sub = flagged.loc[mask].copy()
        res = fit_cohort_models(sub, baselines=mitt_fits[name])
        m_pes.append({"cell": name, "subset": label, **res})

m_pes_df = pd.DataFrame(m_pes)
m_pes_df.to_csv(f"{OUT_DIR}/mittelstaedt_stress2_pes_priming.csv", index=False)
print("\nmittelstaedt stress 2: post-error and alternation controls")
print(m_pes_df[["cell", "subset", "n_trials", "dbic_m1c", "dbic_m3", "dll_m1c_vs_m1", "gamma", "eta", "delta"]].to_string(index=False))

rng = np.random.default_rng(123)
cal_rows = []
n_reps = 15
cal_specs = [
    ("Mittelstaedt_Exp2_Flanker_Speed", mitt_cells["Mittelstaedt_Exp2_Flanker_Speed"], mitt_fits["Mittelstaedt_Exp2_Flanker_Speed"], 0.775, -0.0274),
    ("Hedge_Study2_Flanker_S1", datasets["Hedge_Study2_Flanker_S1"], base_fits["Hedge_Study2_Flanker_S1"], 0.415, -0.0065),
]

for cell_name, tmpl_df, b_fits, g_true, e_true in cal_specs:
    for d_true in (0.0, 0.25, 0.50, 0.70, 0.85):
        wins = 0
        d_hats = []
        gains = []
        for _ in range(n_reps):
            sim_df = sim_cohort_ddm(tmpl_df, b_fits, gamma=g_true, eta=e_true, delta=d_true, rng=rng)
            res = fit_cohort_models(sim_df, baselines=b_fits)
            gain = min(res["dbic_m2"], res["dbic_m3"]) - res["dbic_m1c"]
            if gain < -2.0:
                wins += 1
            d_hats.append(res["delta"])
            gains.append(gain)
        cal_rows.append({
            "cell": cell_name,
            "delta_true": d_true,
            "n_reps": n_reps,
            "power_dbic_lt_neg2": wins / n_reps,
            "mean_delta_hat": float(np.mean(d_hats)),
            "median_vdw_minus_m1c_dbic": float(np.median(gains)),
        })
        print(f"{cell_name} | delta={d_true:.2f} | power={wins}/{n_reps} | mean_hat={np.mean(d_hats):.3f}")

cal_df = pd.DataFrame(cal_rows)
cal_df.to_csv(f"{OUT_DIR}/stress4_delta_power_calibrated.csv", index=False)
print("\nstress 4: calibrated injected-delta power")
print(cal_df.to_string(index=False))

mittelstaedt stress 1: fatigue split
                              cell      slice  n_trials   dbic_m1c    dbic_m3    gamma       eta    delta
Mittelstaedt_Exp2_Flanker_Accuracy early_half      3583 -35.798500 -27.614545 0.944497 -0.022540 0.000000
Mittelstaedt_Exp2_Flanker_Accuracy  late_half      3583 -11.863320  -3.679366 0.748257 -0.001794 0.000000
Mittelstaedt_Exp2_Flanker_Accuracy   Q1_early      1792 -20.094464 -12.603376 1.133842 -0.011898 0.000000
Mittelstaedt_Exp2_Flanker_Accuracy    Q4_late      1791   5.507259  12.997782 0.618610 -0.002147 0.000000
   Mittelstaedt_Exp2_Flanker_Speed early_half      3560 -36.322369 -28.144854 0.710556 -0.007606 0.000000
   Mittelstaedt_Exp2_Flanker_Speed  late_half      3557 -66.912900 -58.736228 0.858186 -0.047011 0.000000
   Mittelstaedt_Exp2_Flanker_Speed   Q1_early      1781 -18.586086 -11.101156 0.747354 -0.011655 0.000000
   Mittelstaedt_Exp2_Flanker_Speed    Q4_late      1777 -24.105661 -16.622979 0.871244 -0.043572 0.000000
  Mittels